<a href="https://colab.research.google.com/github/T2718/Project/blob/main/ec%E3%81%AE%E3%83%AA%E3%82%B9%E3%83%88%E6%93%8D%E4%BD%9C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install selenium

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.6/9.6 MB 42.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.3/510.3 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.6/131.6 kB 5.1 MB/s eta 0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.5.0
    Uninstalling urllib3-2.5.0:
      Successfully uninstalled urllib3-2.5.0


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:

#@title ECのリスト操作

folder = "/content/drive/MyDrive/共有フォルダ/情報/ec" #@param {"type": "string"}
#@markdown <br>
kind = "init" #@param ["init", "spank", "main"]

import pathlib
import re
import requests
from bs4 import BeautifulSoup
import copy
import time
import random
import tkinter as tk
import json
import ast
import datetime

useSelenium = True #@param {type: "boolean"}
importSelenium = True #@param {type: "boolean"}

if useSelenium:
  if importSelenium:
    !pip install selenium
    !pip install webdriver-manager
    !apt-get purge chromium-browser chromium-chromedriver -y
    !wget -q -O - https://dl-ssl.google.com/linux/linux_signing_key.pub | apt-key add -
    !echo 'deb [arch=amd64] http://dl.google.com/linux/chrome/deb/ stable main' | tee /etc/apt/sources.list.d/google-chrome.list
    !apt-get update
    !apt-get install google-chrome-stable -y
  from selenium import webdriver
  from selenium.webdriver.chrome.options import Options
  from selenium.webdriver.chrome.service import Service
  from webdriver_manager.chrome import ChromeDriverManager

  chrome_options = Options()
  chrome_options.add_argument('--headless')
  chrome_options.add_argument('--no-sandbox')
  chrome_options.add_argument(
    "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
  )
  chrome_options.add_argument("--disable-blink-features=AutomationControlled")
  chrome_options.add_argument('--disable-dev-shm-usage')
  chrome_options.binary_location = '/usr/bin/google-chrome'

  # Automatically download and use the correct chromedriver
  service = Service(ChromeDriverManager().install())
  driver = webdriver.Chrome(service=service, options=chrome_options)



folder_path = pathlib.Path(folder)


url_list_path = folder_path / 'url-list.txt'
url_sort_list_path = folder_path / 'url-sort-list.txt'
info_list_path = folder_path / 'info-list.txt'

siteList = {
  'zozovideo.com': {
    'name': 'zozo',
    'code': 0
  },
  'jp.spankbang.com': {
    'name': 'spank',
    'code': 1
  }
}

def makeFile(path):
  if not path.exists():
    path.touch()

def test(text, kind):
  soup = BeautifulSoup(text, 'html.parser')

  if kind == 'extractAllCode':
    for element in soup.find_all(string=True):
      element.extract()
    print(soup.prettify())
  elif kind == 'allCode':
    print(soup.prettify())
  elif kind == 'title':
    element = soup.find('meta', attrs={'name': 'twitter:title'})
    print(element['content'])
  elif kind == 'information':
    information = soup.find('div', attrs={'class': 'information_box'})
    print(information)
  elif kind == 'video':
    video = soup.find(id="video")
    print(video)

def getZozo(soup):
  result = {
    'title': 'Unknown',
    'status': [],
    'date': datetime.date.today(),
    'information': {

    }
  }
  #soup = BeautifulSoup(text, 'html.parser')
  video = soup.find(id='video')

  # 動画が見れない場合
  if not video:
    result['status'].append("Not found Video")
  else:
    poster = video['poster']
    video_src = video.find('source')['src']

    result['poster_url'] = poster
    result['video_url'] = video_src


  information = soup.find('div', {'class': 'information_box'})
  #print(information)

  # 作品情報が載っていない時
  if not information:
    result['status'].append("Not found Information")
  else:
    invalid = False
    for li in information.select('ul li'):
      key = li.select_one('.information-left')
      value = li.select_one('.information-right')
      if key and not value:
        tmp = BeautifulSoup(str(li), "html.parser").li
        k = tmp.select_one('.information-left')
        if k:
            k.decompose()
        value = tmp
      if not key or not value:
        if not invalid:
          result['status'].append('Invalid Information')
          result['information_code'] = str(information)
          invalid = True
          continue
      else:
        key_text = key.get_text(" ", strip=True).replace("：", "")
        value_text = value.get_text(" ", strip=True)
        result['information'][key_text] = value_text
        if key_text == 'タイトル':
          result['title'] = value_text
          result['series_url'] = value.find('a')['href']
  if not result['title']:
    result['title'] = soup.find('meta', attrs={'name': 'twitter:title'})
    if result['title']:
       result['title'] = result['title']['content']
    else:
      result['title'] = soup.title.text
      if not result['title']:
        result['title'] = 'No TITLE'

  return result

def getSpank(url, soup):
  result = {
    'title': 'Unknown',
    'status': [],
    'date': datetime.date.today(),
    'information': {

    }
  }

  def getUrl(result, soup):
    if soup.title.text == 'Just a moment...':
      print("Can't get code")
      result['status'] = "Can't get code"
      return
    main = soup.find('main')
    if not main:
      print("Not found 'main' tag")
      result['status'].append("Not found 'main' tag")
      return
    script = main.find('script')
    if not script:
      print("Not found 'script' tag")
      result['status'].append("Not found 'script' tag")
      return
    urls_re = re.search(r'var stream_data = ({[^\n]+})', script.prettify())
    if not urls_re:
      print("Can't get url")
      result['status'].append("Can't get url")
      return
    urls = urls_re.group(1)
    urls = ast.literal_eval(urls.strip())
    if not 'main' in urls.keys():
      print("Not found main url")
      result['status'].append("Not found main url")
      return
    result['video_url'] = urls['main'][0]
  getUrl(result, soup)

  def getTitle(result, soup):
    # title
    video = soup.find(id='video')
    if not video:
      print("Not found 'video' of id")
      result['status'].append("Not found 'video' of id")
      return

    h1 = video.find('h1')
    if not h1:
      print("Not found 'h1' tag in 'video'")
      result['status'].append("Not found 'h1' tag in 'video'")
      return

    result['title'] = h1.text

  getTitle(result, soup)
  print(result)
  return result

def getBySelenium(url, screenshot=False, ms=1000):
  driver.get(url)
  time.sleep(ms/1000)
  if screenshot:
    w = driver.execute_script("return document.body.scrollWidth;")
    h = driver.execute_script("return document.body.scrollHeight;")

    # set window size
    driver.set_window_size(w,h)

    # Get Screen Shot
    driver.save_screenshot('/content/screenshoot.png')
  html = driver.page_source
  time.sleep(random.uniform(2,4))
  return html

def getHtml(info):
  if info['site'] == 'zozo':
    return requests.get(info['url']).text
  elif info['site'] == 'spank':
    return getBySelenium(info['url'])

def accessAndGetInfo(info):
  global text
  if info['site'] == 'zozo':
    text = getHtml(info);
    soup = BeautifulSoup(text, 'html.parser')
    data = getZozo(soup)
    #print(data)
    return {
      'ok': True,
      'url': info['url'],
      'sitename': info['sitename'],
      'site': 'zozo',
      'data': data
    }
  elif info['site'] == 'spank':
    #text = getBySelenium(info['url'], 1000*random.uniform(1, 1.5))
    #soup = BeautifulSoup(text, 'html.parser')
    #data = getSpank(info['url'], soup)
    #取得が制限されるので一つずつ手動でgetBySeleniumやる！
    global spankList
    spankList.append(info['url'])
    return {
      'ok': True,
      'url': info['url'],
      'status': ['コンソールのボタンを押して一つずつ読み込んでください。'],
      'sitename': info['sitename'],
      'site': 'spank',
      'data': None,
      'date': datetime.date.today()
    }
  return {
    'ok': False,
    'url': info['url'],
    'sitename': info['sitename'],
    'status': ['予想外のサイトの種別です'],
    'site': None,
    'data': None,
    'date': datetime.date.today()
  }

def minProcess(url):
  url = url.strip()
  sitenameRe = r'^https?://([^/]+)'
  if not re.match(sitenameRe, url):
    return {
      'ok': False,
      'content': {
        'url': url,
        'status': ['不正なURLです。']
      }
    }
  else:
    sitename = re.match(sitenameRe, url).group(1)
    site = siteList.get(sitename)
    if not site:
      site = {
        'name': 'other',
        'code': 10
      }

    info = {
      'url': url,
      'sitename': sitename,
      'site': site['name']
    }

    return {
      'ok': True,
      'content': accessAndGetInfo(info),
      'series_code': site['code']
    }

def sort(info_list):
  return sorted(info_list, key=lambda x: (x['series_code'], x['status_code']))

def formatter(info):
  info = copy.deepcopy(info)
  Info = {}
#{'ok': True, 'content': {'ok': True, 'url': 'https://zozovideo.com/nariyuki-papakatsu-girls-episode-1/', 'sitename': 'zozovideo.com', 'site': 'zozo', 'data': {'title': 'なりゆき→パパ活GIRLS!! The Animation #1「J○×オジサマの快感セックスライフ♥」', 'status': [], 'information': {'タイトル': 'なりゆき→パパ活GIRLS!! The Animation #1「J○×オジサマの快感セックスライフ♥」', 'Title': 'Nariyuki Papakatsu Girls!! The Animation Episode 1', '発売日': '2018/02/23', '制作': 'Pink Pineapple'}, 'poster_url': 'https://zozovideo.com/wp-content/uploads/2021/04/359d1ea5136363cd5705ef49ab33d28a.jpg', 'video_url': 'https://hstorage.xyz/files/N/nariyuki-papakatsu-girls-animation/nariyuki-papakatsu-girls-animation-1.mp4', 'series_url': 'https://zozovideo.com/playlist/nariyuki-papakatsu-girls'}}}
  content = info['content']
  if not content:
    return None
  if not info['ok']:
    return {'series_code': 20, 'status_code': 20, 'status': content['status'], 'title': 'Unknown', 'url': {'origin': content['url']}, 'date': datetime.date.today()}
  else:
    if not content['ok']:
      return {'series_code': 20, 'status_code': 10, 'status': content['status'], 'title': 'Unknown', 'url': {'origin': content['url']}, 'date': datetime.date.today()}
    else:
      Info['series_code'] = info['series_code']
      if content['site'] == 'zozo':
        data = content['data']
        status = data['status']
        Info['status_code'] = 10
        Info['status'] = []
        if len(status) == 0:
          Info['status'] = ['成功']
          Info['status_code'] = 0
        elif "Not found Video" in status:
          Info['status'].append('動画が見つかりません')
          Info['status_code'] = 1
        elif "Not found Information" in status:
          Info['status'].append('作品情報が見つかりません')
          Info['status_code'] = 2
        elif 'Invalid Information' in status:
          Info['status'].append('作品情報が不正な型になっています。')
          Info['status_code'] = 3

        Info['title'] = data['title']
        Info['url'] = {
          'origin': content['url']
        }

        if 'video_url' in data.keys():
          Info['url']['video'] = data['video_url']
        if 'poster_url' in data.keys():
          Info['url']['poster'] = data['poster_url']
        if 'series_url' in data.keys():
          Info['series_url'] = data['series_url']

        if 'information' in data.keys():
          Info['information'] = data['information']
      else:
        Info['url'] = {
          'origin': content['url']
        }
        Info['status_code'] = 10
        Info['title'] = 'Unknown'
  return Info

def process(url_list):
  global spankList
  spankList = []
  print('start')
  info_list = []
  for url in url_list:
    info_list.append(formatter(minProcess(url)))
    #print(info_list[-1])
  info_list = sort(info_list)
  for info in info_list:
    print(info)
  print("Write in 'url-sort-list.txt'")
  makeFile(url_sort_list_path)
  with open(url_sort_list_path, 'w') as f:
    for info in info_list:
      f.write(info['title']+'\n - '+info['url']['origin']+'\n')
  #test(text, 'allCode')
  print("Write in 'info-list.txt'")
  makeFile(info_list_path)
  with open(info_list_path, 'w') as f:
    for info in info_list:
      f.write(str(info)+'\n')
  print('finish')

def init():
  if not folder_path.exists():
    raise Exception('folder not found')
  if not url_list_path.exists():
    raise Exception('list file not found')
  with open(url_list_path) as f:
    process(f.readlines())

def getOneSpank():
  global spankIndex
  url = spankList[spankIndex]
  getSpank(url, BeautifulSoup(getBySelenium(url)))
  spankIndex += 1
  if spankIndex == len(spankList):
    print("SpankList終了しました。")

if kind == 'init':
  init()

#while True:
#  input("Enter待ち")
#  getOneSpank()

#test(requests.get('https://zozovideo.com/48671/').text, 'information')

#print(requests.get('https://jp.spankbang.com/4492v/video/hime+sama+4').text)

#text = getBySelenium('https://jp.spankbang.com/3wlzl/video/kyonyuu+fantasy+2', ms=1120)

#test(getBySelenium('https://jp.spankbang.com/3wlzl/video/kyonyuu+fantasy+2'),'allCode')

#test(text, 'allCode')

#url = 'https://jp.spankbang.com/3wlzl/video/kyonyuu+fantasy+2'

def callSpank():
  for i in range(len(spankList)):
    print(str(i)+": "+spankList[i])

  while True:
    inp = input()
    if inp == '':
      return
    url = ""
    try:
      url = spankList[int(inp)]
    except:
      url = inp

    spankSoup = BeautifulSoup(getBySelenium(url))
    getSpank(url, spankSoup)

#callSpank()

if kind == "spank":
  with open("/content/drive/MyDrive/共有フォルダ/情報/ec/info-list.txt") as f:
    info_list = f.readlines()
  spank_info_list = []
  for ind in range(len(info_list)):
    info = ast.literal_eval(info_list[ind].strip())
    if info['series_code'] == 1:
      info['col_num'] = ind
      print(str(len(spank_info_list))+': '+str(info))
      spank_info_list.append(info)

  while True:
    info = input()
    if info == '':
      print('終了')
      break
    info = spank_info_list[int(info)]
    if not info:
      print('Error')
      continue
    if not info['status_code'] == 10:
      print('このURLから情報を取ることはできません。Statusを確認してください。')
      continue
    url = info['url']['origin']
    spankSoup = BeautifulSoup(getBySelenium(url))
    getSpank(url, spankSoup)

#text = getBySelenium('https://jp.spankbang.com/40lib/video/russian+milf+hentai+3')

#with open('/content/index.html', 'w') as f:
#  f.write(text)

#soup = BeautifulSoup(text, 'html.parser')
#print(soup.find_all('video'))

#RE = r'片田舎に嫁いできた'
#print(len(re.findall(RE, text)))
#for match in re.finditer(RE, text):
#  index = match.start()
#  print(text[index-500:index+500])
#  print('\n\n\n')

commentOut = """

--- ZOZO ---
 - 全て成功
{
  'status': '成功',
  'title': 'みだれうち ＃2',
  'sitename': 'zozovideo.com',
  'url': {
    'origin': 'https://zozovideo.com/midareuchi-episode-2/',
    'poster': 'https://zozovideo.com/wp-content/uploads/2024/04/a150372d66e68b79d99a4f3d87713a99.jpg',
    'video': 'https://hstorage.xyz/files/M/midareuchi/midareuchi-2_480p.mp4',
    'series': 'https://zozovideo.com/playlist/midareuchi/'
  },
  'information': {
    'タイトル': 'みだれうち ＃2',
    'Title': 'Midareuchi Episode 2',
    '発売日': '2024/04/26',
    '制作': 'あんてきぬすっ'
  },
}
 - 動画の取得に失敗
{
  'status': '動画の取得に失敗',
  'title': 'みだれうち ＃2',
  'sitename': 'zozovideo.com',
  'url': {
    'origin': 'https://zozovideo.com/midareuchi-episode-2/',
    'poster': 'https://zozovideo.com/wp-content/uploads/2024/04/a150372d66e68b79d99a4f3d87713a99.jpg',
    'video': 'https://hstorage.xyz/files/M/midareuchi/midareuchi-2_480p.mp4',
    'series': 'https://zozovideo.com/playlist/midareuchi/'
  },
  'information': {
    'タイトル': 'みだれうち ＃2',
    'Title': 'Midareuchi Episode 2',
    '発売日': '2024/04/26',
    '制作': 'あんてきぬすっ'
  },
}

 --- spankbang ---
<main class="main-container" data-testid="main">
   <!-- prettier-ignore-start -->
   <script type="text/javascript">
    var ana_video_id = '6560337';
        var stream_data = {'240p': ['https://vdownload-5.sb-cd.com/6/5/6560337-240p.mp4?secure=UFcFzMLAnBFj34_z0loceA,1777691063&m=5&d=6&_tid=6560337'], '320p': [], '480p': ['https://vdownload-5.sb-cd.com/6/5/6560337-480p.mp4?secure=UFcFzMLAnBFj34_z0loceA,1777691063&m=5&d=6&_tid=6560337'], '720p': ['https://vdownload-5.sb-cd.com/6/5/6560337-720p.mp4?secure=UFcFzMLAnBFj34_z0loceA,1777691063&m=5&d=6&_tid=6560337'], '1080p': [], '4k': [], 'mpd': [], 'm3u8': ['https://hls-uranus.sb-cd.com/hls/6/5/6560337-,240p,480p,720p,.mp4.urlset/master.m3u8?secure=UFcFzMLAnBFj34_z0loceA,1777691063&m=5&d=6&_tid=6560337'], 'm3u8_240p': ['https://hls-uranus.sb-cd.com/hls/6/5/6560337-,240p,.mp4.urlset/master.m3u8?secure=UFcFzMLAnBFj34_z0loceA,1777691063&m=5&d=6&_tid=6560337'], 'm3u8_320p': [], 'm3u8_480p': ['https://hls-uranus.sb-cd.com/hls/6/5/6560337-,480p,.mp4.urlset/master.m3u8?secure=UFcFzMLAnBFj34_z0loceA,1777691063&m=5&d=6&_tid=6560337'], 'm3u8_720p': ['https://hls-uranus.sb-cd.com/hls/6/5/6560337-,720p,.mp4.urlset/master.m3u8?secure=UFcFzMLAnBFj34_z0loceA,1777691063&m=5&d=6&_tid=6560337'], 'm3u8_1080p': [], 'm3u8_4k': [], 'cover_image': 'https://tbi.sb-cd.com/t/6560337/db/b8/w:1280/t6-enh/kyonyuu-fantasy-2.jpg', 'thumbnail': 'https://tbi.sb-cd.com/t/6560337/db/b8/w:300/t6-enh/kyonyuu-fantasy-2.jpg', 'stream_raw_id': 6560337, 'stream_sheet': 'https://tbv.sb-cd.com/t/6560337/db/b8/t9999.jpg', 'length': 1790, 'main': ['https://vdownload-5.sb-cd.com/6/5/6560337-720p.mp4?secure=UFcFzMLAnBFj34_z0loceA,1777691063&m=5&d=6&_tid=6560337']};
        var live_keywords = 'hentai,big tits,cartoon,stripchat,demon,kyonyuu fantasy,';



            var ads_channel = 'UGC';
   </script>
   <script type="text/javascript">
    window.dataLayer.push({
            'recommendation_scenario': "2",
        });
   </script>
   <script type="text/javascript">
    window.next_video = ' /83t0u/video/todo+lo+que+quiero+hacer+es+que+me+llenen+la+playa ';
   </script>
   <!-- prettier-ignore-end -->
   <div id="video_theater">
   </div>
   <div class="main_content_container">
    <div data-streamkey="NjU2MDMzNw.jH_RQO9MwkrvNyjI3-FMXMsuBAQ" data-testid="video-page" data-videoid="6560337" id="video" x-data="videoPage">
     <div class="left flex flex-col gap-2">
      <div class="flex flex-row gap-4">
       <h1 class="main_content_title" title="巨乳ファンタジー2">
        巨乳ファンタジー2
       </h1>
       <span class="responsive-page views">
        <svg class="i_svg i_new-ui-visible-outlined">
         <use xlink:href="#new-ui-visible-outlined">
         </use>
        </svg>
        94,945 ビュー
       </span>
      </div>
      <div class="relative z-0" x-data="horizontalScroller('', '', undefined)">
       <span @click="scrollLeft()" @mousedown.prevent="if ($event.detail &gt; 1) $event.preventDefault();" class="before:from-kobalt-700 absolute top-0 -left-1 z-10 flex h-full cursor-pointer items-center before:pointer-events-none before:absolute before:top-0 before:left-0 before:h-full before:w-24 before:bg-gradient-to-r" style="display: none;" x-show="showPrev" x-transition="">
        <svg class="i_svg i_chevron-left">
         <use xlink:href="#chevron-left">
         </use>

"""

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
Package 'chromium-browser' is not installed, so not removed
Package 'chromium-chromedriver' is not installed, so not removed
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.
OK
deb [arch=amd64] http://dl.google.com/linux/chrome/deb/ stable main
Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:2 http://dl.google.com/linux/chrome/deb stable InRelease [1,825 B]
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:4 https://cli.github.com/packages stable InRelease [3,917 B]
Get:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:6 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:9 http://dl.google.com/linux/chrome/deb stable/main amd64 Packages [1,200 B

Exception: folder not found

In [ ]:
import requests
from urllib.parse import urlparse
import pathlib

def download_mp4(url, filename=None):
    if filename is None:
        # Extract filename from URL, removing query parameters
        parsed_url = urlparse(url)
        filename = parsed_url.path.split('/')[-1]
        if not filename:
            filename = 'downloaded_video' # Default if no filename in path
        if not filename.lower().endswith('.mp4'):
            filename += '.mp4'

    # Ensure filename is unique to prevent overwriting
    original_filename = filename
    counter = 1
    while pathlib.Path(filename).exists():
        name_without_ext, ext = original_filename.rsplit('.', 1) if '.' in original_filename else (original_filename, '')
        filename = f"{name_without_ext}_{counter}.{ext}" if ext else f"{name_without_ext}_{counter}"
        counter += 1

    print(f"Downloading {url} to {filename}...")
    try:
        response = requests.get(url, stream=True)
        response.raise_for_status() # Raise an exception for bad status codes
        with open(filename, 'wb') as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
        print(f"Successfully downloaded {filename}")
        return filename
    except requests.exceptions.RequestException as e:
        print(f"Error downloading {url}: {e}")
        return None

video_url_from_user = 'https://vdownload-5.sb-cd.com/6/5/6560337-720p.mp4?secure=UFcFzMLAnBFj34_z0loceA,1777691063&m=5&d=6&_tid=6560337'
downloaded_file = download_mp4(video_url_from_user, filename='downloaded_from_user.mp4')
if downloaded_file:
    print(f"Video saved as: {downloaded_file}")
else:
    print("Video download failed.")

Successfully downloaded downloaded_from_user.mp4
Video saved as: downloaded_from_user.mp4


In [ ]:

driver.get('https://mathotagamma.github.io/APIs/CompVisJS/1-02-02/CompVisJS.js')

html = driver.page_source
print(html)

<html><head><meta name="color-scheme" content="light dark"></head><body><pre style="word-wrap: break-word; white-space: pre-wrap;">//変更点:ViewとViewThreeのaddGraphの引数の構造を変更し、それぞれのgraphの管理をidにした。また、ViewにaddArrowメソッドを追加し、CompVisにgetIdをstaticで,deleteIdをメソッドで追加。また、View,ViewThreeにはdeleteGraph(id)を追加し、ViewThreeにgetState(id=undefined)を追加。
//注意点
/*
例:
viewer.addGraph(
  (t) =&gt; 
    {
      return t**2 - 4*t + 5;
    }
  ,{ start: -3, end: 5, samples: 1200, color: "#00aaff" }
);

必ずアロー関数は
(変数) =&gt; { return ~~~ ;}
で書くこと。('{}','return',';'必須)
*/
/*
CompVis.ViewThreeはTHREE.jsを使用しています。
※内部に組み込んでいるため、別で用意する必要はありません。
THREE : "https://cdn.jsdelivr.net/npm/three@0.152.2/build/three.module.js",
OrbitControls : "https://cdn.jsdelivr.net/npm/three@0.152.2/examples/jsm/controls/OrbitControls.js",
CSS2DRenderer : "https://cdn.jsdelivr.net/npm/three@0.152.2/examples/jsm/renderers/CSS2DRenderer.js"

使用例
const myDiv = document.getElementById("my-div");
const viewThree = new CompVis.ViewThree(myDiv);

// 非同期で